# 1. Definitions and compilation

A pipeline definition is portable intent. It names inputs, procedure-backed nodes, bindings, outputs, and metadata. It deliberately does not contain live Python implementations. Compilation resolves those names through Provium catalogs and proves the graph is coherent before any durable run exists.


## A representative definition

```yaml
identifier: tutorial.text
version: 1.0.0
inputs:
  document:
    artifact: example.DocumentV1
    scope: record
    cardinality: one
nodes:
  tokenize:
    procedure: example.TokenizeV1
    inputs:
      source: $inputs.document
outputs:
  tokens: $nodes.tokenize.destination
```

References make dataflow explicit. `$inputs.*` reads the frozen run input snapshot; `$nodes.*.*` reads a named upstream node output. This lets the compiler derive edges rather than trusting a separately maintained dependency list.


In [1]:
from provium_pipeline import (
    PipelineIdentifier,
    PipelineInputName,
    PipelineNodeIdentifier,
    PipelineOutputName,
    PipelineVersion,
)
from provium_pipeline.definition import parse_reference

identity = PipelineIdentifier('tutorial.text')
version = PipelineVersion('1.0.0')
names = (
    PipelineInputName('document'), PipelineNodeIdentifier('tokenize'),
    PipelineOutputName('tokens'),
)
input_reference = parse_reference('$inputs.document')
node_reference = parse_reference('$nodes.tokenize.destination')
assert str(identity) == 'tutorial.text' and str(version) == '1.0.0'
identity, version, names, input_reference, node_reference


('tutorial.text',
 '1.0.0',
 ('document', 'tokenize', 'tokens'),
 PipelineInputReference(name='document'),
 NodeOutputReference(node='tokenize', output='destination'))

## What compilation contributes

`PipelineCompiler` resolves artifact and procedure definitions, validates binding names and artifact compatibility, rejects cycles, orders nodes topologically, constructs output contracts, resolves configuration layers, and emits canonical digests. The digest identifies the resolved meaning—not merely the source YAML text.

Configuration is layered so package defaults, site configuration, and run-specific overrides remain distinguishable. The resolved configuration snapshot is attached to the run; changing a config file later cannot rewrite history.


In [2]:
from provium_pipeline.compiler import (
    CompiledPipeline,
    PipelineCompilationError,
    PipelineCompiler,
    PipelineConfigurationLayer,
    ResolvedPipelineConfiguration,
)

compiler_contract = {
    'compiler': PipelineCompiler,
    'result': CompiledPipeline,
    'configuration_layer': PipelineConfigurationLayer,
    'resolved_configuration': ResolvedPipelineConfiguration,
    'failure': PipelineCompilationError,
}
assert all(compiler_contract.values())
compiler_contract


{'compiler': provium_pipeline.compiler.compiler.PipelineCompiler,
 'result': provium_pipeline.compiler.models.CompiledPipeline,
 'configuration_layer': provium_pipeline.compiler.resolution.PipelineConfigurationLayer,
 'resolved_configuration': provium_pipeline.compiler.resolution.ResolvedPipelineConfiguration,
 'failure': provium_pipeline.compiler.diagnostics.PipelineCompilationError}

## Validation workflow

Use `provium pipeline validate pipeline.yaml` for diagnostics without creating state, `provium pipeline show pipeline.yaml` for the canonical definition, and `provium pipeline list` for discovered catalog entries. Keep definitions under version control; treat compiled snapshots and digests as run evidence.

**What to notice:** builders improve authoring ergonomics, codecs define portable documents, catalogs resolve names, and compilation is the semantic firewall before orchestration. Next: [inputs and runs](02-inputs-and-runs.ipynb). Reference: [definitions and compilation](../docs/definitions-and-compilation.md).
